In [1]:
import zipfile
import os

zip_path = "fraud_full_features.zip"
extract_path = "/content/fraud_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("압축 해제 완료!")

압축 해제 완료!


In [2]:
for root, dirs, files in os.walk(extract_path):
    for file in files:
        print(os.path.join(root, file))

/content/fraud_data/fraud_full_features.csv


In [3]:
import pandas as pd
import glob

csv_files = glob.glob(
    os.path.join(extract_path, "**", "*.csv"),
    recursive=True
)

print("발견된 CSV 파일:")
for file in csv_files:
    print(file)

if len(csv_files) == 0:
    raise FileNotFoundError("ZIP 파일 안에서 CSV 파일을 찾지 못했습니다.")

csv_path = csv_files[0]

df = pd.read_csv(csv_path)

print("\n데이터 불러오기 완료!")
print("사용한 파일:", csv_path)
print("데이터 크기:", df.shape)

display(df.head())

발견된 CSV 파일:
/content/fraud_data/fraud_full_features.csv

데이터 불러오기 완료!
사용한 파일: /content/fraud_data/fraud_full_features.csv
데이터 크기: (1296675, 31)


,trans_date_trans_time,cc_num,merchant,category,amt,is_fraud,recent_24h_high_amt_count,category_recent_fraud_rate,category_recent_fraud_rate_missing,count_30min,...,amt_to_prior_median_ratio,is_10x_prior_median,has_prior_normal_transaction,outside_trans_hours_80,is_online,risk_time_22_04,interact_repeat_category,merchant_change_count,rolling_sum_amt_1h,is_high_amt
0,2019-01-01 12:47:15,60416207185,"fraud_Jones, Sawayn and Romaguera",misc_net,7.27,0,0,0.000000,0,0,...,NaN,0,0,0,1,0,0.0,0,7.27,0
1,2019-01-02 08:44:57,60416207185,fraud_Berge LLC,gas_transport,52.94,0,0,0.006154,0,1,...,7.281981,0,1,0,0,0,0.0,1,52.94,0
2,2019-01-02 08:47:36,60416207185,fraud_Luettgen PLC,gas_transport,82.08,0,0,0.006135,0,2,...,2.726457,0,1,0,0,0,0.0,1,135.02,0
3,2019-01-02 12:38:14,60416207185,fraud_Daugherty LLC,kids_pets,34.79,0,0,0.000000,0,1,...,0.657159,0,1,0,0,0,0.0,1,34.79,0
4,2019-01-02 13:10:46,60416207185,fraud_Beier and Sons,home,27.18,0,0,0.000000,0,1,...,0.619628,0,1,0,0,0,0.0,1,61.97,0


In [4]:
print("총 변수 개수:", len(df.columns))

for i, col in enumerate(df.columns, start=1):
    print(f"{i:2d}. {col}")

총 변수 개수: 31
 1. trans_date_trans_time
 2. cc_num
 3. merchant
 4. category
 5. amt
 6. is_fraud
 7. recent_24h_high_amt_count
 8. category_recent_fraud_rate
 9. category_recent_fraud_rate_missing
10. count_30min
11. Repeat3
12. high_speed
13. speed_2
14. customer_mean_amt
15. customer_std_amt
16. amt_ratio_to_mean
17. amt_zscore_card
18. customer_transaction_count
19. trans_hour
20. age
21. prior_normal_median_amt
22. amt_to_prior_median_ratio
23. is_10x_prior_median
24. has_prior_normal_transaction
25. outside_trans_hours_80
26. is_online
27. risk_time_22_04
28. interact_repeat_category
29. merchant_change_count
30. rolling_sum_amt_1h
31. is_high_amt


In [5]:
cluster_features = [
    "amt",
    "trans_hour",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "count_30min"
]

required_cols = (
    ["trans_date_trans_time", "is_fraud"]
    + cluster_features
)

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    print("❌ 없는 변수:")
    print(missing_cols)
else:
    print("✅ 필요한 변수들이 모두 존재합니다.")

✅ 필요한 변수들이 모두 존재합니다.


In [6]:
df["trans_date_trans_time"] = pd.to_datetime(
    df["trans_date_trans_time"]
)

df = (
    df
    .sort_values("trans_date_trans_time")
    .reset_index(drop=True)
)

print("전체 데이터 크기:", df.shape)

print(
    "거래 기간:",
    df["trans_date_trans_time"].min(),
    "~",
    df["trans_date_trans_time"].max()
)

print(
    "이상거래 비율:",
    df["is_fraud"].mean()
)

전체 데이터 크기: (1296675, 31)
거래 기간: 2019-01-01 00:00:18 ~ 2020-06-21 12:13:37
이상거래 비율: 0.005788651743883394


fraud_full_features.zip
        ↓
압축 해제
        ↓
fraud_full_features.csv
        ↓
1,296,675 × 31
        ↓
시간형 변환
        ↓
시간순 정렬
        ↓
K-means용 7개 행동변수 존재 확인
        ↓
여기까지 완료 ✅

TP = 3,385
FP = 136
FN = 294
TN = 644,523

In [7]:
# ============================================================
# STEP 7-1. LightGBM 조합3 최종 설정
# ============================================================

import numpy as np
import pandas as pd

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

TARGET = "is_fraud"
TIME_COL = "trans_date_trans_time"

RANDOM_STATE = 42
EARLY_STOPPING_ROUNDS = 100


# 최종 변수 조합 3
FINAL_FEATURES = [
    "category",
    "amt",
    "trans_hour",
    "age",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "prior_normal_median_amt",
    "count_30min",
    "high_speed"
]

CATEGORICAL_FEATURES = [
    "category"
]


# 최종 LightGBM 튜닝 결과
BEST_LGBM_PARAMS = {
    "num_leaves": 23,
    "max_depth": 5,
    "min_child_samples": 150,

    "learning_rate": 0.03,

    "subsample": 0.9,
    "subsample_freq": 1,
    "colsample_bytree": 0.9,

    "reg_alpha": 0.1,
    "reg_lambda": 1.0,

    "min_split_gain": 0.0,

    "max_bin": 255
}


# 최종 OOF F1 최대 임계값
FINAL_THRESHOLD = 0.9289392017223173


print("조합3 변수 수:", len(FINAL_FEATURES))
print("최종 Threshold:", FINAL_THRESHOLD)

조합3 변수 수: 11
최종 Threshold: 0.9289392017223173


In [8]:
df[TIME_COL] = pd.to_datetime(
    df[TIME_COL]
)

df = (
    df
    .sort_values(TIME_COL)
    .reset_index(drop=True)
)

print("전체 데이터:", df.shape)

print(
    "기간:",
    df[TIME_COL].min(),
    "~",
    df[TIME_COL].max()
)

전체 데이터: (1296675, 31)
기간: 2019-01-01 00:00:18 ~ 2020-06-21 12:13:37


In [9]:
# ============================================================
# Expanding Window 3-Fold
# ============================================================

number_of_rows = len(df)

boundaries = np.linspace(
    0,
    number_of_rows,
    7,
    dtype=int
)

time_folds = []

for fold_number in range(1, 4):

    train_end = boundaries[
        fold_number + 2
    ]

    validation_start = train_end

    validation_end = boundaries[
        fold_number + 3
    ]

    time_folds.append({

        "fold": fold_number,

        "train_idx": np.arange(
            0,
            train_end
        ),

        "val_idx": np.arange(
            validation_start,
            validation_end
        )
    })


for fold in time_folds:

    print(
        f"Fold {fold['fold']} | "
        f"Train={len(fold['train_idx']):,} | "
        f"Validation={len(fold['val_idx']):,}"
    )

Fold 1 | Train=648,337 | Validation=216,113
Fold 2 | Train=864,450 | Validation=216,112
Fold 3 | Train=1,080,562 | Validation=216,113


In [10]:
def prepare_lgbm_fold(
    source_df,
    train_indices,
    validation_indices
):

    # ---------------------------
    # X
    # ---------------------------

    X_train = (
        source_df.iloc[
            train_indices
        ][FINAL_FEATURES]
        .copy()
        .reset_index(drop=True)
    )

    X_val = (
        source_df.iloc[
            validation_indices
        ][FINAL_FEATURES]
        .copy()
        .reset_index(drop=True)
    )


    # ---------------------------
    # y
    # ---------------------------

    y_train = (
        source_df.iloc[
            train_indices
        ][TARGET]
        .astype(np.int8)
        .reset_index(drop=True)
    )

    y_val = (
        source_df.iloc[
            validation_indices
        ][TARGET]
        .astype(np.int8)
        .reset_index(drop=True)
    )


    # ---------------------------
    # category 처리
    # Train category 기준으로
    # Validation도 통일
    # ---------------------------

    for col in CATEGORICAL_FEATURES:

        train_categories = (
            X_train[col]
            .astype("string")
            .fillna("missing")
            .unique()
            .tolist()
        )

        X_train[col] = pd.Categorical(
            X_train[col]
            .astype("string")
            .fillna("missing"),
            categories=train_categories
        )

        X_val[col] = pd.Categorical(
            X_val[col]
            .astype("string")
            .fillna("missing"),
            categories=train_categories
        )

    return (
        X_train,
        X_val,
        y_train,
        y_val
    )

In [11]:
# ============================================================
# 조합3 3-Fold OOF 예측
# ============================================================

oof_parts = []

fold_results = []

for fold_info in time_folds:

    fold_number = fold_info["fold"]

    train_idx = fold_info["train_idx"]
    val_idx = fold_info["val_idx"]

    (
        X_train,
        X_val,
        y_train,
        y_val
    ) = prepare_lgbm_fold(
        source_df=df,
        train_indices=train_idx,
        validation_indices=val_idx
    )


    # Fold Train 기준 클래스 가중치
    negative_count = int(
        (y_train == 0).sum()
    )

    positive_count = int(
        (y_train == 1).sum()
    )

    scale_pos_weight = (
        negative_count
        / positive_count
    )


    print()
    print("=" * 70)
    print(f"Fold {fold_number}")
    print("=" * 70)

    print(
        f"Train={len(X_train):,} | "
        f"Validation={len(X_val):,} | "
        f"scale_pos_weight="
        f"{scale_pos_weight:.6f}"
    )


    # -----------------------------
    # LightGBM
    # -----------------------------

    model = LGBMClassifier(

        objective="binary",

        # Early Stopping용 충분한 최대값
        n_estimators=2500,

        **BEST_LGBM_PARAMS,

        scale_pos_weight=
            scale_pos_weight,

        random_state=
            RANDOM_STATE,

        n_jobs=-1,
        verbosity=-1
    )


    # -----------------------------
    # 학습
    # -----------------------------

    model.fit(

        X_train,
        y_train,

        eval_set=[
            (
                X_val,
                y_val
            )
        ],

        eval_metric=
            "average_precision",

        categorical_feature=
            CATEGORICAL_FEATURES,

        callbacks=[

            early_stopping(
                stopping_rounds=
                    EARLY_STOPPING_ROUNDS,

                first_metric_only=True,
                verbose=False
            ),

            log_evaluation(
                period=0
            )
        ]
    )


    # -----------------------------
    # Validation 확률
    # -----------------------------

    val_probability = (
        model.predict_proba(
            X_val,
            num_iteration=
                model.best_iteration_
        )[:, 1]
    )


    fold_pr_auc = (
        average_precision_score(
            y_val,
            val_probability
        )
    )


    # -----------------------------
    # OOF 저장
    # -----------------------------

    fold_oof = pd.DataFrame({

        "original_index":
            val_idx,

        "fold":
            fold_number,

        "y_true":
            y_val.to_numpy(),

        "y_probability":
            val_probability
    })


    oof_parts.append(
        fold_oof
    )


    fold_results.append({

        "Fold":
            fold_number,

        "Train":
            len(X_train),

        "Validation":
            len(X_val),

        "Best Iteration":
            model.best_iteration_,

        "PR-AUC":
            fold_pr_auc
    })


    print(
        f"Best iteration="
        f"{model.best_iteration_}"
    )

    print(
        f"PR-AUC="
        f"{fold_pr_auc:.6f}"
    )


Fold 1
Train=648,337 | Validation=216,113 | scale_pos_weight=168.411288
Best iteration=1973
PR-AUC=0.973007

Fold 2
Train=864,450 | Validation=216,112 | scale_pos_weight=174.772672
Best iteration=1303
PR-AUC=0.980234

Fold 3
Train=1,080,562 | Validation=216,113 | scale_pos_weight=171.036618
Best iteration=1432
PR-AUC=0.977862


In [12]:
oof_df = (
    pd.concat(
        oof_parts,
        ignore_index=True
    )
    .sort_values(
        "original_index"
    )
    .reset_index(drop=True)
)

fold_result_df = pd.DataFrame(
    fold_results
)

display(fold_result_df)

print()
print(
    "OOF 거래 수:",
    f"{len(oof_df):,}"
)

print(
    "OOF 이상거래 수:",
    f"{int(oof_df['y_true'].sum()):,}"
)

print(
    "OOF 전체 PR-AUC:",
    f"{average_precision_score(
        oof_df['y_true'],
        oof_df['y_probability']
    ):.6f}"
)

,Fold,Train,Validation,Best Iteration,PR-AUC
0,1,648337,216113,1973,0.973007
1,2,864450,216112,1303,0.980234
2,3,1080562,216113,1432,0.977862



OOF 거래 수: 648,338
OOF 이상거래 수: 3,679
OOF 전체 PR-AUC: 0.976489


In [13]:
oof_df["y_pred"] = (
    oof_df[
        "y_probability"
    ]
    >= FINAL_THRESHOLD
).astype(np.int8)

In [14]:
conditions = [

    # TP
    (
        (oof_df["y_true"] == 1)
        &
        (oof_df["y_pred"] == 1)
    ),

    # FP
    (
        (oof_df["y_true"] == 0)
        &
        (oof_df["y_pred"] == 1)
    ),

    # FN
    (
        (oof_df["y_true"] == 1)
        &
        (oof_df["y_pred"] == 0)
    ),

    # TN
    (
        (oof_df["y_true"] == 0)
        &
        (oof_df["y_pred"] == 0)
    )
]


labels = [
    "TP",
    "FP",
    "FN",
    "TN"
]


oof_df["error_type"] = np.select(
    conditions,
    labels,
    default="Unknown"
)

In [15]:
error_counts = (
    oof_df[
        "error_type"
    ]
    .value_counts()
)

display(error_counts)

,count
error_type,
TN,644468
TP,3432
FN,247
FP,191


In [16]:
oof_transactions = (
    df.iloc[
        oof_df[
            "original_index"
        ].to_numpy()
    ]
    .copy()
    .reset_index(drop=True)
)

# 예측 결과 추가
oof_transactions[
    "oof_fold"
] = oof_df["fold"].to_numpy()

oof_transactions[
    "pred_probability"
] = oof_df[
    "y_probability"
].to_numpy()

oof_transactions[
    "pred_label"
] = oof_df[
    "y_pred"
].to_numpy()

oof_transactions[
    "error_type"
] = oof_df[
    "error_type"
].to_numpy()

In [17]:
tp_df = (
    oof_transactions[
        oof_transactions[
            "error_type"
        ] == "TP"
    ]
    .copy()
)

fp_df = (
    oof_transactions[
        oof_transactions[
            "error_type"
        ] == "FP"
    ]
    .copy()
)

fn_df = (
    oof_transactions[
        oof_transactions[
            "error_type"
        ] == "FN"
    ]
    .copy()
)

tn_df = (
    oof_transactions[
        oof_transactions[
            "error_type"
        ] == "TN"
    ]
    .copy()
)


print("TP:", len(tp_df))
print("FP:", len(fp_df))
print("FN:", len(fn_df))
print("TN:", len(tn_df))

TP: 3432
FP: 191
FN: 247
TN: 644468


핵심은 TP·FP·FN·TN은 데이터 자체에 원래 존재하는 속성이 아니라, LightGBM의 예측 결과와 실제 정답을 비교해야 생기는 분류라는 거야.

예를 들어 원본 데이터에는 기본적으로:

거래 정보 + 실제 정답 is_fraud

가 있어. 하지만 아직 TP, FP, FN, TN은 없어.

LightGBM이 해당 거래를 예측해야 비로소:

실제 is_fraud	LightGBM 예측	결과
1	1	TP
1	0	FN
0	1	FP
0	0	TN

이렇게 정해져.

그리고 지금 사용한 시간순 Expanding Window에서는 전체 1,296,675건 중 648,338건만 학습에 포함되지 않은 상태에서 OOF 예측을 받았어.

그래서:

전체 거래 1,296,675건
│
├─ 최초 학습구간 648,337건
│   └─ OOF 예측 없음
│      → TP/FP/FN/TN 없음
│
└─ OOF 예측구간 648,338건
    └─ LightGBM 예측 있음
       ↓
       실제 is_fraud와 비교
       ↓
       TP / FP / FN / TN

따라서 지금 분석에서

TP + FP + FN + TN = 648,338건

이 되는 게 맞아.

다만 “전체 데이터 건수로 절대 만들 수 없다”는 뜻은 아니야. 전체 1,296,675건을 LightGBM으로 예측하면 형식적으로는 전부 TP/FP/FN/TN으로 나눌 수 있어.

문제는 최초 648,337건은 이미 LightGBM이 학습할 때 정답까지 본 데이터이기 때문에, 그것을 다시 예측해서 TP/FP/FN/TN으로 분류하면 공정한 오류 분석이 아니게 된다는 것이야.

그래서 지금 목적이 “LightGBM이 처음 보는 거래에서 무엇을 잘 잡고, 무엇을 놓치고, 무엇을 오탐하는가?”라면 현재의 OOF 648,338건을 대상으로 TP/FP/FN/TN → 각각 K-means 하는 게 맞아.

즉 한 문장으로 정리하면:

K-means 때문에 전체 데이터를 못 쓰는 게 아니라, TP·FP·FN·TN이라는 라벨 자체가 신뢰할 수 있는 LightGBM 예측이 존재하는 OOF 거래에서만 만들어졌기 때문에 현재 648,338건을 사용하는 것이다.

이 구분이 아주 중요해. 전체 거래 K-means라면 1,296,675건 전부 사용 가능하고, TP/FP/FN/TN별 K-means라면 현재는 OOF 648,338건을 사용하는 것이야.

In [18]:
tp_df.to_csv(
    "TP_transactions.csv",
    index=False
)

fp_df.to_csv(
    "FP_transactions.csv",
    index=False
)

fn_df.to_csv(
    "FN_transactions.csv",
    index=False
)

tn_df.to_csv(
    "TN_transactions.csv",
    index=False
)

print("✅ TP/FP/FN/TN 저장 완료")

✅ TP/FP/FN/TN 저장 완료


In [19]:
CLUSTER_FEATURES = [
    "amt",
    "trans_hour",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "count_30min"
]

In [20]:
fn_cluster_raw = (
    fn_df[
        CLUSTER_FEATURES
    ]
    .copy()
)

print(
    "FN 거래 수:",
    len(fn_cluster_raw)
)

display(
    fn_cluster_raw.describe().T
)

FN 거래 수: 247


,count,mean,std,min,25%,50%,75%,max
amt,247.0,254.536154,357.638938,4.060000,17.560000,22.720000,505.675000,1276.770000
trans_hour,247.0,16.028340,7.618122,0.000000,12.000000,20.000000,22.000000,23.000000
recent_24h_high_amt_count,247.0,0.417004,0.686835,0.000000,0.000000,0.000000,1.000000,4.000000
amt_to_prior_median_ratio,244.0,6.799089,10.376030,0.094402,0.345172,0.639682,10.400689,50.295040
rolling_sum_amt_1h,247.0,321.771417,418.510865,5.990000,19.145000,50.530000,662.970000,1984.970000
amt_zscore_card,247.0,1.359933,3.807896,-32.771658,-0.452940,-0.225167,2.978060,15.183974
count_30min,247.0,0.927126,0.593626,0.000000,1.000000,1.000000,1.000000,3.000000


In [21]:
from sklearn.preprocessing import StandardScaler


def preprocess_error_group(
    group_df
):

    X = (
        group_df[
            CLUSTER_FEATURES
        ]
        .copy()
    )


    # =================================
    # 1. inf → NaN
    # =================================

    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )


    # =================================
    # 2. 결측치 중앙값 대체
    # =================================

    for col in CLUSTER_FEATURES:

        X[col] = (
            X[col]
            .fillna(
                X[col].median()
            )
        )


    # =================================
    # 3. 양의 왜도 변수 log1p
    # =================================

    log_features = [
        "amt",
        "recent_24h_high_amt_count",
        "amt_to_prior_median_ratio",
        "rolling_sum_amt_1h"
    ]

    for col in log_features:

        X[col] = np.log1p(
            X[col]
        )


    # =================================
    # 4. Z-score 변수 Signed Log
    # =================================

    X[
        "amt_zscore_card"
    ] = (
        np.sign(
            X[
                "amt_zscore_card"
            ]
        )
        *
        np.log1p(
            np.abs(
                X[
                    "amt_zscore_card"
                ]
            )
        )
    )


    # =================================
    # 5. StandardScaler
    # =================================

    scaler = StandardScaler()

    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=X.columns,
        index=X.index
    )


    return (
        X,
        X_scaled,
        scaler
    )

In [22]:
(
    fn_transformed,
    fn_scaled,
    fn_scaler
) = preprocess_error_group(
    fn_df
)

print(
    "FN 결측치:",
    fn_scaled.isna().sum().sum()
)

print(
    "FN inf:",
    np.isinf(
        fn_scaled
    ).sum().sum()
)

FN 결측치: 0
FN inf: 0


In [23]:
from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)


fn_k_results = []

fn_models = {}

for k in range(2, 5):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(
        fn_scaled
    )

    fn_models[k] = model

    counts = pd.Series(
        labels
    ).value_counts()

    fn_k_results.append({

        "K": k,

        "Silhouette":
            silhouette_score(
                fn_scaled,
                labels
            ),

        "Davies_Bouldin":
            davies_bouldin_score(
                fn_scaled,
                labels
            ),

        "Calinski_Harabasz":
            calinski_harabasz_score(
                fn_scaled,
                labels
            ),

        "Smallest Cluster":
            int(counts.min()),

        "Largest Cluster":
            int(counts.max()),

        "Largest Cluster (%)":
            counts.max()
            / len(labels)
            * 100
    })


fn_k_evaluation = pd.DataFrame(
    fn_k_results
)

display(fn_k_evaluation)

,K,Silhouette,Davies_Bouldin,Calinski_Harabasz,Smallest Cluster,Largest Cluster,Largest Cluster (%)
0,2,0.466434,0.879582,249.552975,85,162,65.587045
1,3,0.379303,1.122347,189.224696,68,96,38.866397
2,4,0.344869,1.202775,164.032311,43,90,36.437247


In [24]:
fn_best_k = int(
    fn_k_evaluation
    .sort_values(
        "Silhouette",
        ascending=False
    )
    .iloc[0]["K"]
)

print(
    "Silhouette 기준 FN 후보 K:",
    fn_best_k
)

Silhouette 기준 FN 후보 K: 2


In [25]:
(
    fp_transformed,
    fp_scaled,
    fp_scaler
) = preprocess_error_group(
    fp_df
)

print(
    "FP 거래 수:",
    len(fp_scaled)
)

print(
    "FP 결측치:",
    fp_scaled.isna().sum().sum()
)

print(
    "FP inf:",
    np.isinf(
        fp_scaled
    ).sum().sum()
)

FP 거래 수: 191
FP 결측치: 0
FP inf: 0


In [26]:
fp_k_results = []

fp_models = {}

for k in range(2, 5):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(
        fp_scaled
    )

    fp_models[k] = model

    counts = pd.Series(
        labels
    ).value_counts()

    fp_k_results.append({

        "K": k,

        "Silhouette":
            silhouette_score(
                fp_scaled,
                labels
            ),

        "Davies_Bouldin":
            davies_bouldin_score(
                fp_scaled,
                labels
            ),

        "Calinski_Harabasz":
            calinski_harabasz_score(
                fp_scaled,
                labels
            ),

        "Smallest Cluster":
            int(counts.min()),

        "Largest Cluster":
            int(counts.max()),

        "Largest Cluster (%)":
            counts.max()
            / len(labels)
            * 100
    })


fp_k_evaluation = pd.DataFrame(
    fp_k_results
)

display(fp_k_evaluation)

,K,Silhouette,Davies_Bouldin,Calinski_Harabasz,Smallest Cluster,Largest Cluster,Largest Cluster (%)
0,2,0.494555,0.846170,229.656747,94,97,50.785340
1,3,0.467791,0.859471,192.646779,34,87,45.549738
2,4,0.436360,0.876818,184.452492,34,62,32.460733


In [27]:
fp_best_k = int(
    fp_k_evaluation
    .sort_values(
        "Silhouette",
        ascending=False
    )
    .iloc[0]["K"]
)

print(
    "Silhouette 기준 FP 후보 K:",
    fp_best_k
)

Silhouette 기준 FP 후보 K: 2


In [28]:
# ============================================================
# STEP 11-1. TP 전처리
# ============================================================

(
    tp_transformed,
    tp_scaled,
    tp_scaler
) = preprocess_error_group(tp_df)

print("TP 거래 수:", len(tp_scaled))
print("TP 결측치:", tp_scaled.isna().sum().sum())
print("TP inf:", np.isinf(tp_scaled).sum().sum())

TP 거래 수: 3432
TP 결측치: 0
TP inf: 0


In [29]:
# ============================================================
# STEP 11-2. TP K=2~6 탐색
# ============================================================

tp_k_results = []
tp_models = {}

for k in range(2, 7):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(tp_scaled)

    tp_models[k] = model

    counts = pd.Series(labels).value_counts()

    tp_k_results.append({
        "K": k,

        "Silhouette":
            silhouette_score(tp_scaled, labels),

        "Davies_Bouldin":
            davies_bouldin_score(tp_scaled, labels),

        "Calinski_Harabasz":
            calinski_harabasz_score(tp_scaled, labels),

        "Smallest Cluster":
            int(counts.min()),

        "Largest Cluster":
            int(counts.max()),

        "Largest Cluster (%)":
            counts.max() / len(labels) * 100
    })


tp_k_evaluation = pd.DataFrame(tp_k_results)

display(tp_k_evaluation)

,K,Silhouette,Davies_Bouldin,Calinski_Harabasz,Smallest Cluster,Largest Cluster,Largest Cluster (%)
0,2,0.443443,0.996021,2131.726090,648,2784,81.118881
1,3,0.324717,1.280745,2022.108587,605,1624,47.319347
2,4,0.347632,1.213173,1756.309691,478,1515,44.143357
3,5,0.283496,1.314386,1591.253799,469,904,26.340326
4,6,0.290311,1.312110,1512.706918,206,846,24.650350


In [30]:
tp_best_k = int(
    tp_k_evaluation
    .sort_values(
        "Silhouette",
        ascending=False
    )
    .iloc[0]["K"]
)

print(
    "Silhouette 기준 TP 후보 K:",
    tp_best_k
)

Silhouette 기준 TP 후보 K: 2


In [31]:
# ============================================================
# STEP 12-1. TN 전처리
# ============================================================

(
    tn_transformed,
    tn_scaled,
    tn_scaler
) = preprocess_error_group(tn_df)

print("TN 거래 수:", len(tn_scaled))
print("TN 결측치:", tn_scaled.isna().sum().sum())
print("TN inf:", np.isinf(tn_scaled).sum().sum())

TN 거래 수: 644468
TN 결측치: 0
TN inf: 0


In [32]:
# ============================================================
# STEP 12-2. TN Silhouette 평가용 동일 표본 생성
# ============================================================

SILHOUETTE_SAMPLE_SIZE = min(
    30000,
    len(tn_scaled)
)

rng = np.random.RandomState(42)

tn_sample_idx = rng.choice(
    len(tn_scaled),
    size=SILHOUETTE_SAMPLE_SIZE,
    replace=False
)

print(
    "Silhouette 평가 표본:",
    len(tn_sample_idx)
)

Silhouette 평가 표본: 30000


In [33]:
# ============================================================
# STEP 12-3. TN K=2~8 탐색
# ============================================================

tn_k_results = []
tn_models = {}

for k in range(2, 9):

    print(f"K={k} 진행 중...")

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(tn_scaled)

    tn_models[k] = model

    counts = pd.Series(labels).value_counts()

    # 같은 30,000건으로 Silhouette 평가
    sample_labels = labels[tn_sample_idx]

    silhouette = silhouette_score(
        tn_scaled.iloc[tn_sample_idx],
        sample_labels
    )

    tn_k_results.append({
        "K": k,

        "Silhouette":
            silhouette,

        "Davies_Bouldin":
            davies_bouldin_score(
                tn_scaled,
                labels
            ),

        "Calinski_Harabasz":
            calinski_harabasz_score(
                tn_scaled,
                labels
            ),

        "Smallest Cluster":
            int(counts.min()),

        "Largest Cluster":
            int(counts.max()),

        "Largest Cluster (%)":
            counts.max() / len(labels) * 100
    })


tn_k_evaluation = pd.DataFrame(tn_k_results)

display(tn_k_evaluation)

K=2 진행 중...
K=3 진행 중...
K=4 진행 중...
K=5 진행 중...
K=6 진행 중...
K=7 진행 중...
K=8 진행 중...


,K,Silhouette,Davies_Bouldin,Calinski_Harabasz,Smallest Cluster,Largest Cluster,Largest Cluster (%)
0,2,0.324689,1.196725,321581.627092,286844,357624,55.491351
1,3,0.353991,0.987942,274804.120184,24765,345251,53.571473
2,4,0.305349,1.077649,274496.107920,24541,309589,48.037917
3,5,0.304949,1.050894,276447.289438,24476,194499,30.179776
4,6,0.311972,1.053731,267267.612387,24433,182315,28.289225
5,7,0.312559,1.045736,258725.068880,15254,170927,26.522186
6,8,0.328476,1.030547,257855.273749,14773,155394,24.111981


In [34]:
tn_best_k = int(
    tn_k_evaluation
    .sort_values(
        "Silhouette",
        ascending=False
    )
    .iloc[0]["K"]
)

print(
    "Silhouette 기준 TN 후보 K:",
    tn_best_k
)

Silhouette 기준 TN 후보 K: 3


In [35]:
# ============================================================
# STEP 13. TP / FP / FN / TN별 최적 K 한눈에 비교
# 기준: Silhouette Score 최대
# ============================================================

import pandas as pd

evaluation_dict = {
    "TP": tp_k_evaluation,
    "FP": fp_k_evaluation,
    "FN": fn_k_evaluation,
    "TN": tn_k_evaluation
}

best_k_results = []

for group, result_df in evaluation_dict.items():

    # Silhouette가 가장 높은 행 선택
    best_row = (
        result_df
        .sort_values("Silhouette", ascending=False)
        .iloc[0]
    )

    best_k_results.append({
        "Group": group,
        "Best K": int(best_row["K"]),
        "Silhouette": best_row["Silhouette"],
        "Davies-Bouldin": best_row["Davies_Bouldin"],
        "Calinski-Harabasz": best_row["Calinski_Harabasz"],
        "Smallest Cluster": int(best_row["Smallest Cluster"]),
        "Largest Cluster": int(best_row["Largest Cluster"]),
        "Largest Cluster (%)": best_row["Largest Cluster (%)"]
    })


best_k_summary = pd.DataFrame(best_k_results)

display(
    best_k_summary.style.format({
        "Silhouette": "{:.4f}",
        "Davies-Bouldin": "{:.4f}",
        "Calinski-Harabasz": "{:.2f}",
        "Largest Cluster (%)": "{:.2f}%"
    })
)

,Group,Best K,Silhouette,Davies-Bouldin,Calinski-Harabasz,Smallest Cluster,Largest Cluster,Largest Cluster (%)
0,TP,2,0.4434,0.9960,2131.73,648,2784,81.12%
1,FP,2,0.4946,0.8462,229.66,94,97,50.79%
2,FN,2,0.4664,0.8796,249.55,85,162,65.59%
3,TN,3,0.3540,0.9879,274804.12,24765,345251,53.57%


In [36]:
# ============================================================
# 후보 K 기준 군집 생성
# ※ 아직 최종 K 확정 아님
# ============================================================

from sklearn.cluster import KMeans

CANDIDATE_K = {
    "TP": 2,
    "FP": 2,
    "FN": 2,
    "TN": 3
}

scaled_data_dict = {
    "TP": tp_scaled,
    "FP": fp_scaled,
    "FN": fn_scaled,
    "TN": tn_scaled
}

original_data_dict = {
    "TP": tp_df.copy(),
    "FP": fp_df.copy(),
    "FN": fn_df.copy(),
    "TN": tn_df.copy()
}

candidate_models = {}
candidate_cluster_data = {}

for group in ["TP", "FP", "FN", "TN"]:

    k = CANDIDATE_K[group]

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(
        scaled_data_dict[group]
    )

    temp_df = original_data_dict[group].copy()

    temp_df["candidate_cluster"] = labels

    candidate_models[group] = model
    candidate_cluster_data[group] = temp_df

    print(
        f"{group}: K={k}, "
        f"N={len(temp_df):,}"
    )

TP: K=2, N=3,432
FP: K=2, N=191
FN: K=2, N=247
TN: K=3, N=644,468


In [37]:
# ============================================================
# 후보 K 군집 크기 확인
# ============================================================

cluster_size_results = []

for group, group_df in candidate_cluster_data.items():

    counts = (
        group_df["candidate_cluster"]
        .value_counts()
        .sort_index()
    )

    for cluster_id, count in counts.items():

        cluster_size_results.append({
            "Group": group,
            "Candidate K": CANDIDATE_K[group],
            "Cluster": int(cluster_id),
            "Count": int(count),
            "Ratio (%)": count / len(group_df) * 100
        })

cluster_size_df = pd.DataFrame(
    cluster_size_results
)

display(
    cluster_size_df.style.format({
        "Ratio (%)": "{:.2f}%"
    })
)

,Group,Candidate K,Cluster,Count,Ratio (%)
0,TP,2,0,2784,81.12%
1,TP,2,1,648,18.88%
2,FP,2,0,97,50.79%
3,FP,2,1,94,49.21%
4,FN,2,0,85,34.41%
5,FN,2,1,162,65.59%
6,TN,3,0,345251,53.57%
7,TN,3,1,24765,3.84%
8,TN,3,2,274452,42.59%


In [38]:
CLUSTER_FEATURES = [
    "amt",
    "trans_hour",
    "recent_24h_high_amt_count",
    "amt_to_prior_median_ratio",
    "rolling_sum_amt_1h",
    "amt_zscore_card",
    "count_30min"
]

In [39]:
# ============================================================
# 후보 군집별 행동변수 평균
# ============================================================

profile_results = []

for group, group_df in candidate_cluster_data.items():

    profile = (
        group_df
        .groupby("candidate_cluster")[CLUSTER_FEATURES]
        .mean()
        .round(3)
        .reset_index()
    )

    profile.insert(
        0,
        "Group",
        group
    )

    profile_results.append(profile)


candidate_cluster_profiles = pd.concat(
    profile_results,
    ignore_index=True
)

display(candidate_cluster_profiles)

,Group,candidate_cluster,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
0,TP,0,687.983,14.741,1.781,16.092,1179.878,4.899,0.652
1,TP,1,24.510,11.435,1.866,0.549,326.569,-0.878,1.157
2,FP,0,33.719,14.732,0.948,0.784,93.524,-0.315,0.938
3,FP,1,903.356,11.947,0.415,25.959,987.579,6.592,0.223
4,FN,0,686.272,12.800,0.165,18.263,736.570,4.963,0.576
5,FN,1,28.008,17.722,0.549,0.670,104.130,-0.531,1.111
6,TN,0,110.446,11.444,0.000,2.544,123.160,0.329,0.962
7,TN,1,69.571,13.094,1.072,1.507,137.890,-0.031,0.949
8,TN,2,13.573,14.482,0.000,0.315,22.111,-0.436,0.853


In [40]:
# ============================================================
# 후보 군집별 행동변수 중앙값
# ============================================================

median_results = []

for group, group_df in candidate_cluster_data.items():

    profile = (
        group_df
        .groupby("candidate_cluster")[CLUSTER_FEATURES]
        .median()
        .round(3)
        .reset_index()
    )

    profile.insert(
        0,
        "Group",
        group
    )

    median_results.append(profile)


candidate_cluster_medians = pd.concat(
    median_results,
    ignore_index=True
)

display(candidate_cluster_medians)

,Group,candidate_cluster,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
0,TP,0,785.770,22.0,2.0,15.345,949.110,4.353,1.0
1,TP,1,13.915,6.5,2.0,0.359,20.925,-0.431,1.0
2,FP,0,18.940,22.0,1.0,0.407,19.800,-0.431,1.0
3,FP,1,836.750,14.0,0.0,23.278,864.935,6.024,0.0
4,FN,0,739.440,14.0,0.0,17.448,739.500,4.472,1.0
5,FN,1,19.380,22.0,0.0,0.407,20.550,-0.345,1.0
6,TN,0,77.600,12.0,0.0,1.742,85.030,0.054,1.0
7,TN,1,49.470,14.0,1.0,0.988,63.790,-0.195,1.0
8,TN,2,8.500,16.0,0.0,0.218,10.360,-0.427,1.0


In [41]:
def show_cluster_profile(group):

    group_df = candidate_cluster_data[group]

    # 건수
    counts = (
        group_df["candidate_cluster"]
        .value_counts()
        .sort_index()
    )

    # 평균
    means = (
        group_df
        .groupby("candidate_cluster")[CLUSTER_FEATURES]
        .mean()
    )

    # 중앙값
    medians = (
        group_df
        .groupby("candidate_cluster")[CLUSTER_FEATURES]
        .median()
    )

    print("=" * 70)
    print(
        f"{group} 후보 K = {CANDIDATE_K[group]}"
    )
    print("=" * 70)

    print("\n[군집 크기]")
    display(
        pd.DataFrame({
            "Count": counts,
            "Ratio (%)":
                counts / len(group_df) * 100
        }).round(2)
    )

    print("\n[행동변수 평균]")
    display(
        means.round(3)
    )

    print("\n[행동변수 중앙값]")
    display(
        medians.round(3)
    )

In [42]:
show_cluster_profile("TP")

TP 후보 K = 2

[군집 크기]


,Count,Ratio (%)
candidate_cluster,,
0,2784,81.12
1,648,18.88



[행동변수 평균]


,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
candidate_cluster,,,,,,,
0,687.983,14.741,1.781,16.092,1179.878,4.899,0.652
1,24.510,11.435,1.866,0.549,326.569,-0.878,1.157



[행동변수 중앙값]


,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
candidate_cluster,,,,,,,
0,785.770,22.0,2.0,15.345,949.110,4.353,1.0
1,13.915,6.5,2.0,0.359,20.925,-0.431,1.0


### TP 군집 프로파일 해석 — 후보 K = 2

**TP(True Positive)** 는 실제 이상거래이며, LightGBM도 이상거래라고 **정확하게 탐지한 거래**이다.  
총 3,432건의 TP 거래는 K-means를 통해 **Cluster 0(2,784건, 81.12%)** 과 **Cluster 1(648건, 18.88%)** 의 두 유형으로 구분되었다.

#### Cluster 0 — 고액·평소 대비 금액 급증형

Cluster 0은 **거래금액 자체가 크고, 고객의 평소 거래금액과 비교해도 매우 큰 금액이 결제된 유형**으로 해석할 수 있다.

- 거래금액(`amt`)의 중앙값이 **785.77**로 Cluster 1의 13.92보다 매우 높다.
- 평소 거래 중앙값 대비 현재 거래금액의 비율(`amt_to_prior_median_ratio`)도 중앙값이 **15.35배**로, 평소보다 훨씬 큰 금액이 거래되었다.
- 카드 기준 거래금액 이상도(`amt_zscore_card`) 역시 중앙값 **4.35**로 높아, 해당 카드의 일반적인 거래금액 범위에서 크게 벗어난 거래임을 보여준다.
- 최근 1시간 누적 거래금액(`rolling_sum_amt_1h`)도 중앙값 **949.11**로 매우 높다.
- 최근 24시간 고액거래 횟수의 중앙값도 **2회**이다.

즉, **"평소보다 갑자기 매우 큰 금액을 사용하면서 단기간 누적 거래금액도 커진 이상거래"** 가 이 군집의 핵심 특징이다. 전체 TP의 약 81%가 이 군집에 포함된다는 점을 고려하면, LightGBM이 정확하게 잡아낸 이상거래의 상당수가 이러한 **뚜렷한 금액 이상 패턴**을 가지고 있다고 볼 수 있다.

#### Cluster 1 — 소액·반복거래형

Cluster 1은 Cluster 0과 달리 **거래금액 자체는 작고 평소 거래금액에서도 크게 벗어나지 않지만, 비교적 반복적인 거래 행동을 보이는 유형**이다.

- 거래금액 중앙값은 **13.92**로 매우 낮다.
- 평소 대비 거래금액 비율의 중앙값도 **0.36배**로, 평소보다 오히려 작은 금액이다.
- `amt_zscore_card` 중앙값 역시 **-0.43**으로 금액 측면에서는 일반적인 고액 이상거래와 반대되는 모습을 보인다.
- 반면 최근 24시간 고액거래 횟수의 중앙값은 Cluster 0과 동일한 **2회**이다.
- 최근 30분 거래횟수(`count_30min`) 평균은 **1.16회**로 Cluster 0의 0.65회보다 높다.

따라서 이 군집은 **"현재 거래 자체는 소액이지만 주변 시점에서 반복적인 거래 행동이 나타나는 이상거래"** 로 해석할 수 있다.

#### 종합 해석

TP가 두 군집으로 나뉜 결과는 LightGBM이 이상거래를 탐지할 때 **한 가지 패턴만 잡고 있는 것은 아니라는 점** 을 보여준다.  
하나는 **금액이 비정상적으로 커지는 유형**, 다른 하나는 **현재 금액은 작지만 반복적인 거래 행동이 나타나는 유형**이다.

특히 두 군집 사이에서 금액 관련 변수의 차이가 매우 뚜렷하므로, **TP를 K=2로 구분하는 것은 행동 측면에서도 충분한 해석 가능성을 가진 후보**라고 볼 수 있다.

In [43]:
show_cluster_profile("FP")

FP 후보 K = 2

[군집 크기]


,Count,Ratio (%)
candidate_cluster,,
0,97,50.79
1,94,49.21



[행동변수 평균]


,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
candidate_cluster,,,,,,,
0,33.719,14.732,0.948,0.784,93.524,-0.315,0.938
1,903.356,11.947,0.415,25.959,987.579,6.592,0.223



[행동변수 중앙값]


,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
candidate_cluster,,,,,,,
0,18.94,22.0,1.0,0.407,19.800,-0.431,1.0
1,836.75,14.0,0.0,23.278,864.935,6.024,0.0


### FP 군집 프로파일 해석 — 후보 K = 2

**FP(False Positive)** 는 실제로는 정상거래이지만 LightGBM이 **이상거래라고 잘못 판단한 거래**, 즉 오탐이다.  
총 191건은 **Cluster 0(97건, 50.79%)** 과 **Cluster 1(94건, 49.21%)** 로 거의 절반씩 나뉘었다.

#### Cluster 0 — 소액·반복 정상거래 오탐형

Cluster 0은 **금액 자체는 크지 않지만 최근 거래 활동이 반복적으로 나타나는 정상거래**에 가깝다.

- 거래금액 중앙값은 **18.94**로 낮다.
- 평소 대비 거래금액 비율도 중앙값 **0.41배**로 평소보다 오히려 작은 거래이다.
- 카드 기준 금액 이상도 역시 중앙값 **-0.43**으로 금액 자체가 비정상적으로 높다고 보기 어렵다.
- 반면 최근 24시간 고액거래 횟수 중앙값은 **1회**이다.
- 최근 30분 거래횟수도 중앙값 **1회**이다.

따라서 이 군집은 **"금액은 평범하거나 작은데 최근 거래 활동이 존재하여 모델이 이상거래로 오인한 정상거래"** 로 볼 수 있다.

#### Cluster 1 — 고액·평소 대비 급증 정상거래 오탐형

Cluster 1은 정상거래임에도 **이상거래와 매우 비슷한 고액·금액 이탈 패턴**을 보인다.

- 거래금액 중앙값이 **836.75**로 매우 높다.
- 평소 대비 거래금액 비율은 무려 **23.28배**이다.
- 카드 기준 거래금액 Z-score도 중앙값 **6.02**로 매우 높다.
- 최근 1시간 누적 거래금액 역시 중앙값 **864.94**로 높다.
- 반면 최근 24시간 고액거래 횟수의 중앙값은 **0회**, 최근 30분 거래횟수도 **0회**이다.

즉, 반복거래라기보다는 **"갑자기 한 번 매우 큰 금액이 발생한 정상거래"** 에 가깝다.

이러한 거래는 실제로는 정상이지만 행동변수만 보면 이상거래와 매우 유사하기 때문에 LightGBM이 오탐했을 가능성이 있다.

#### 종합 해석

FP는 매우 흥미롭게 **191건이 거의 정확하게 두 유형으로 나뉘었다.**

1. **금액은 작지만 거래 활동성이 나타나는 정상거래**
2. **반복성은 낮지만 평소보다 매우 큰 금액을 결제한 정상거래**

즉, LightGBM의 오탐은 하나의 원인에 집중되어 있지 않고 **'반복적인 행동'과 '급격한 고액 결제'라는 서로 다른 정상 행동을 이상거래로 혼동하는 두 가지 패턴**으로 나타난다.

두 군집의 크기도 50.79%와 49.21%로 매우 균형적이고 행동 차이도 뚜렷하므로, **FP에서는 K=2가 특히 해석하기 좋은 후보**라고 판단할 수 있다.


In [44]:
show_cluster_profile("FN")

FN 후보 K = 2

[군집 크기]


,Count,Ratio (%)
candidate_cluster,,
0,85,34.41
1,162,65.59



[행동변수 평균]


,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
candidate_cluster,,,,,,,
0,686.272,12.800,0.165,18.263,736.57,4.963,0.576
1,28.008,17.722,0.549,0.670,104.13,-0.531,1.111



[행동변수 중앙값]


,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
candidate_cluster,,,,,,,
0,739.44,14.0,0.0,17.448,739.50,4.472,1.0
1,19.38,22.0,0.0,0.407,20.55,-0.345,1.0


### FN 군집 프로파일 해석 — 후보 K = 2

**FN(False Negative)** 은 실제로는 이상거래인데 LightGBM이 **정상거래라고 판단하여 놓친 거래**이다.  
FDS 관점에서는 실제 사기를 통과시킨 경우이므로 특히 중요한 분석 대상이다.

총 247건은 **Cluster 0(85건, 34.41%)** 과 **Cluster 1(162건, 65.59%)** 로 나뉘었다.

#### Cluster 0 — 고액·평소 대비 급증형 미탐지 거래

Cluster 0은 모델이 놓친 거래임에도 **금액 측면에서 상당히 강한 이상 징후**를 가지고 있다.

- 거래금액 중앙값이 **739.44**로 높다.
- 평소 대비 거래금액 비율도 중앙값 **17.45배**이다.
- 카드 기준 금액 Z-score도 중앙값 **4.47**로 높다.
- 최근 1시간 누적 거래금액 역시 중앙값 **739.50**으로 높다.

반면 최근 24시간 고액거래 횟수 중앙값은 **0회**이다.

즉, **"평소보다 갑자기 매우 큰 금액이 발생했지만 최근에 비슷한 고액거래가 반복되지는 않은 이상거래"** 로 볼 수 있다.

금액 이상 신호는 강하지만 반복성 신호가 약한 것이 특징이며, 이러한 일부 거래가 LightGBM에서 FN으로 남았다는 점을 추가적으로 살펴볼 필요가 있다.

#### Cluster 1 — 정상거래처럼 보이는 소액·반복형 미탐지 거래

Cluster 1은 반대로 **금액만 보면 이상거래라고 판단하기 상당히 어려운 유형**이다.

- 거래금액 중앙값은 **19.38**에 불과하다.
- 평소 대비 거래금액 비율도 중앙값 **0.41배**이다.
- 카드 기준 금액 Z-score도 중앙값 **-0.35**로 낮다.
- 최근 1시간 누적 거래금액도 중앙값 **20.55**로 낮다.
- 반면 최근 30분 거래횟수 평균은 **1.11회**로 Cluster 0의 0.58회보다 높다.

즉, **"금액 측면에서는 평소 거래와 크게 다르지 않아 정상처럼 보이지만, 상대적으로 거래 반복성이 나타나는 이상거래"** 라고 해석할 수 있다.

전체 FN의 **65.59%가 이 유형**이라는 점은 중요하다. 모델이 놓친 이상거래 중 상당수가 단순한 고액 이상거래가 아니라 **정상적인 소비처럼 보이는 소액 거래**라는 것을 보여주기 때문이다.

#### 종합 해석

FN에서도 두 가지 상당히 다른 유형이 확인된다.

1. **고액이지만 반복성이 낮아 놓친 유형**
2. **금액이 정상적이어서 놓친 소액·반복 유형**

특히 두 번째 유형이 FN의 약 66%를 차지한다는 점에서, LightGBM의 주요 미탐지 영역 중 하나는 **금액 이상성이 뚜렷하지 않은 이상거래**일 가능성이 있다.

따라서 FN의 K=2 역시 단순히 통계적으로 두 군집으로 나뉜 것뿐만 아니라, **"왜 모델이 사기를 놓쳤는가?"라는 질문을 설명할 수 있는 행동적 차이**를 보여주는 후보라고 판단할 수 있다.

In [45]:
show_cluster_profile("TN")

TN 후보 K = 3

[군집 크기]


,Count,Ratio (%)
candidate_cluster,,
0,345251,53.57
1,24765,3.84
2,274452,42.59



[행동변수 평균]


,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
candidate_cluster,,,,,,,
0,110.446,11.444,0.000,2.544,123.160,0.329,0.962
1,69.571,13.094,1.072,1.507,137.890,-0.031,0.949
2,13.573,14.482,0.000,0.315,22.111,-0.436,0.853



[행동변수 중앙값]


,amt,trans_hour,recent_24h_high_amt_count,amt_to_prior_median_ratio,rolling_sum_amt_1h,amt_zscore_card,count_30min
candidate_cluster,,,,,,,
0,77.60,12.0,0.0,1.742,85.03,0.054,1.0
1,49.47,14.0,1.0,0.988,63.79,-0.195,1.0
2,8.50,16.0,0.0,0.218,10.36,-0.427,1.0


### TN 군집 프로파일 해석 — 후보 K = 3

**TN(True Negative)** 은 실제 정상거래이며 LightGBM도 **정상거래라고 정확하게 판단한 거래**이다.  
총 644,468건으로 네 그룹 중 가장 많으며, K-means에서는 **3개의 정상거래 행동유형**으로 구분되었다.

- Cluster 0: **345,251건 (53.57%)**
- Cluster 1: **24,765건 (3.84%)**
- Cluster 2: **274,452건 (42.59%)**

#### Cluster 0 — 일반적인 중간금액 정상거래형

Cluster 0은 세 군집 가운데 상대적으로 거래금액과 평소 대비 금액 수준이 높은 **일반적인 중간금액 정상거래**로 볼 수 있다.

- 거래금액 중앙값은 **77.60**이다.
- 평소 대비 거래금액 비율은 중앙값 **1.74배**이다.
- 최근 1시간 누적금액은 중앙값 **85.03**이다.
- 카드 기준 Z-score는 중앙값 **0.05**로 거의 0에 가깝다.
- 최근 24시간 고액거래는 중앙값 **0회**이다.

즉 현재 금액이 평소보다 어느 정도 높을 수는 있지만, 카드 전체 거래분포에서 극단적인 수준은 아니며 **일상적으로 발생할 수 있는 정상적인 중간 규모 거래**로 해석할 수 있다.

#### Cluster 1 — 최근 고액거래 활동이 있는 정상거래형

Cluster 1은 전체 TN 중 **3.84%로 가장 작은 군집**이지만 다른 두 군집과 구별되는 특징이 있다.

- 거래금액 중앙값은 **49.47**이다.
- 최근 24시간 고액거래 횟수 중앙값이 **1회**로, 다른 두 TN 군집의 0회와 차이가 난다.
- 평소 대비 거래금액 비율은 중앙값 **0.99배**로 거의 평소 수준이다.
- 카드 기준 금액 Z-score도 **-0.20**으로 크게 이례적이지 않다.

따라서 **"최근 고액거래 활동은 있었지만 현재 거래 자체는 고객의 평소 소비 수준과 유사한 정상거래"** 로 해석할 수 있다.

이 군집이 전체 정상거래에서 차지하는 비율은 작지만, `recent_24h_high_amt_count`라는 행동 특성에 의해 다른 정상거래와 구별되는 집단이 형성되었다는 점에서 K=3의 해석 가능성을 높여준다.

#### Cluster 2 — 전형적인 소액 정상거래형

Cluster 2는 **소액이고 평소 소비 수준보다도 작은 거래**가 중심인 유형이다.

- 거래금액 중앙값은 **8.50**으로 세 TN 군집 중 가장 낮다.
- 평소 대비 거래금액 비율도 중앙값 **0.22배**로 매우 낮다.
- 최근 1시간 누적금액도 중앙값 **10.36**으로 낮다.
- 카드 기준 금액 Z-score는 중앙값 **-0.43**이다.
- 최근 24시간 고액거래도 중앙값 **0회**이다.

즉 **"평소보다 작은 금액을 사용하는 전형적인 일상 소액거래"** 라고 해석할 수 있으며, 전체 TN의 42.59%를 차지한다.

#### 종합 해석

TN에서는 정상거래가 단순히 하나의 동일한 형태가 아니라 다음과 같은 **세 가지 행동유형**으로 구분되었다.

1. **일반적인 중간금액 정상거래형** — 53.57%
2. **최근 고액거래 활동이 있는 정상거래형** — 3.84%
3. **전형적인 소액 정상거래형** — 42.59%

특히 Cluster 1은 규모가 작지만 최근 24시간 고액거래 이력이 있다는 명확한 차별점이 있고, Cluster 0과 Cluster 2는 거래금액 및 평소 대비 금액 수준에서 차이가 나타난다.

따라서 TN의 K=3은 **정상거래를 '중간금액형–최근 고액활동형–소액형'으로 세분화할 수 있다는 점에서 해석 가능한 후보**이다. 다만 TN은 다른 그룹보다 Silhouette가 낮았으므로, 향후 안정성 검증에서 이 세 군집 구조가 Seed 변화에도 유지되는지 추가 확인할 필요가 있다.

TP K=2
→ Cluster 0과 1이 행동적으로 명확하게 다름?
→ YES → K=2 지지

FP K=2
→ 94 vs 97 + 행동 특성도 다름?
→ YES → K=2 강하게 지지

FN K=2
→ 85 vs 162 + 놓치는 사기 유형이 두 종류로 설명됨?
→ YES → K=2 지지

TN K=3
→ 세 군집이 각각 다른 정상거래 행동인가?
→ YES → K=3 지지


---

→ NO → K=2 등 차선 후보 재검토